# Volume change with restarts

Using the monthly data for this. 

In [1]:
import glob
import dask.distributed as dsk
import matplotlib.pyplot as plt
import xarray as xr
import cf_xarray as cfxr
import numpy as np
import xesmf as xe
from matplotlib.colors import to_rgba

import cmocean as cm
import cartopy.crs as ccrs

import cartopy.feature as cft
import matplotlib.path as mpath

import warnings # ignore these warnings
warnings.filterwarnings("ignore", category = FutureWarning)
warnings.filterwarnings("ignore", category = UserWarning)
warnings.filterwarnings("ignore", category = RuntimeWarning)

In [44]:
lat_slice = slice(-79,-59)

# Importing restart files
paths_R = np.sort(glob.glob(
    '/scratch/x77/kc5856/access-om2/archive/01deg_jra55v13_ryf9091_21mbath_copy-15025726/restart039/ocean/ocean_velocity*.nc'))
paths_C = np.sort(glob.glob(
    '/scratch/x77/kc5856/access-om2/archive/01deg_jra55v13_ryf9091_21mbath_copy-15025726/restart040/ocean/ocean_thickness*.nc'))
paths_M = np.sort(glob.glob(
    'scratch/x77/kc5856/access-om2/archive/access-ssp126-sofia-expt-c085baf0/restart040/ocean/ocean_thickness*.nc'))

In [38]:
test = xr.open_mfdataset(paths_R)
test # using ocean_thickness

<xarray.Dataset> Size: 35GB
Dimensions:  (Time: 1, zaxis_1: 75, yaxis_1: 2700, xaxis_1: 3600, zaxis_2: 76)
Coordinates:
  * Time     (Time) float64 8B 1.0
  * zaxis_1  (zaxis_1) float64 600B 1.0 2.0 3.0 4.0 5.0 ... 72.0 73.0 74.0 75.0
  * yaxis_1  (yaxis_1) float64 22kB 1.0 2.0 3.0 ... 2.698e+03 2.699e+03 2.7e+03
  * xaxis_1  (xaxis_1) float64 29kB 1.0 2.0 3.0 ... 3.598e+03 3.599e+03 3.6e+03
  * zaxis_2  (zaxis_2) float64 608B 1.0 2.0 3.0 4.0 5.0 ... 73.0 74.0 75.0 76.0
Data variables:
    rho_dzt  (Time, zaxis_1, yaxis_1, xaxis_1) float64 6GB dask.array<chunksize=(1, 19, 135, 180), meta=np.ndarray>
    dzt_dst  (Time, zaxis_1, yaxis_1, xaxis_1) float64 6GB dask.array<chunksize=(1, 19, 135, 180), meta=np.ndarray>
    dstlo    (Time, zaxis_1, yaxis_1, xaxis_1) float64 6GB dask.array<chunksize=(1, 19, 135, 180), meta=np.ndarray>
    dstup    (Time, zaxis_1, yaxis_1, xaxis_1) float64 6GB dask.array<chunksize=(1, 19, 135, 180), meta=np.ndarray>
    dswt     (Time, zaxis_2, yaxis_1, xaxis_1) float64 6GB dask.array<chunksize=(1, 19, 135, 180), meta=np.ndarray>
    dst      (Time, zaxis_1, yaxis_1, xaxis_1) float64 6GB dask.array<chunksize=(1, 19, 135, 180), meta=np.ndarray>
    pbot0    (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
Attributes:
    filename:  ocean_thickness.res.nc
    history:   20230428T1652Z: /g/data/ik11/inputs/access-om2/bin/mppnccombin...

In [31]:
test = xr.open_mfdataset(paths_R)
test # from ocean_sbc

<xarray.Dataset> Size: 467MB
Dimensions:  (Time: 1, yaxis_1: 2700, xaxis_1: 3600)
Coordinates:
  * Time     (Time) float64 8B 1.0
  * yaxis_1  (yaxis_1) float64 22kB 1.0 2.0 3.0 ... 2.698e+03 2.699e+03 2.7e+03
  * xaxis_1  (xaxis_1) float64 29kB 1.0 2.0 3.0 ... 3.598e+03 3.599e+03 3.6e+03
Data variables:
    t_surf   (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
    s_surf   (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
    u_surf   (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
    v_surf   (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
    sea_lev  (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
    frazil   (Time, yaxis_1, xaxis_1) float64 78MB dask.array<chunksize=(1, 540, 720), meta=np.ndarray>
Attributes:
    filename:  ocean_sbc.res.nc
    history:   20230428T1652Z: /g/data/ik11/inputs/access-om2/bin/mppnccombin...

In [8]:
# Time variant variables
def preprocess(ds):
    ds = ds[['dzt']].sel(yt_ocean=lat_slice)
    return ds

data_start = xr.open_mfdataset(paths_R, preprocess = preprocess, chunks = 'auto')
data_C = xr.open_mfdataset(paths_C, preprocess = preprocess, chunks = 'auto')
data_M = xr.open_mfdataset(paths_M, preprocess = preprocess, chunks = 'auto')

KeyError: 'dzt'

In [ ]:
## volume
dzt = cc.querying.getvar(control,'dzt',session,ncfile= 'ocean.nc')
dzt_end = dzt.sel(time = slice('1947-05','1947-06')).mean(dim = 'time')
dzt_start = dzt.sel(time = slice('1937-05','1937-06')).mean(dim = 'time')
volume_start =  dzt_start * dyt * dxt
volume_end =  dzt_end * dyt * dxt
volume_start_shelf, shelf_mask = shelf_mask_isobath(volume_start)
volume_end_shelf, shelf_mask = shelf_mask_isobath(volume_end)